# 第 7 題｜傳統機器學習與模型評估：保單續保流失風險

本 Notebook 使用 Python 標準函式庫與固定亂數種子建立純合成資料。資料不是真實保單、個資或實際續保紀錄；不連網、不讀取外部檔案，也不呼叫服務。

## 合成資料與時間切分

共產生 300 筆資料：月份 0–9、每月 30 筆。保費調整依月份使用平均數 0.05／0.10／0.16、標準差 0.045 的常態抽樣並裁切至 [0, 0.35]；客服聯絡次數 0–4 的抽樣權重為 [35, 30, 20, 10, 5]，年資均勻抽樣於 [0.5, 15]。標籤由指定 logit 的 sigmoid 機率抽樣。

依時間原順序切分 train=前 180 筆、valid=接續 60 筆、test=最後 60 筆，不洗牌。標準化平均數與母體標準差只用訓練集估計；遇到零標準差則使用 1。

## 邏輯斯迴歸、閾值與評估

模型使用全批次梯度下降 1200 步、learning rate=0.12、L2=0.002（不懲罰截距）。只根據驗證集 F1 從 0.10 至 0.90、每次增加 0.05 選閾值；同分時採較高閾值。測試集只用已選定閾值評估。Baseline 機率是訓練集正例率，分類閾值固定為 0.50。以下程式碼與可執行套件共用相同實作。

## 結果解讀、安全與限制

測試資料位於較晚月份，合成保費調整分布隨月份改變，提醒我們需監控資料／概念漂移，不能把一次切分的表現外推至真實業務。誤報會消耗客服容量；閾值應另考量服務量能、聯繫成本與漏報代價。Brier score 單獨不足以證明校準良好，仍需按時間／群體檢查可靠度圖、校準及外部驗證。

模型只能作為人工客服排序的教學示範，不得用於保費、核保、理賠或其他個人權益決策，也不得宣稱合成結果代表真實續保效益。若未來處理真實資料，須先取得授權並採資料最小化、去識別、存取控制及安全保存措施；本 Notebook 不包含也不使用真實保單或個資。

In [1]:
#!/usr/bin/env python3
"""以標準函式庫示範保單續保流失風險的合成資料分類流程。"""

import math
import random

SEED = 20261007
FEATURES = ("premium_change", "contacts_90d", "tenure_years")
THRESHOLDS = [i / 100 for i in range(10, 91, 5)]
TRAIN_STEPS = 1200
LEARNING_RATE = 0.12
L2 = 0.002


def sigmoid(value):
    """數值穩定的 sigmoid。"""
    if value >= 0:
        exp_value = math.exp(-value)
        return 1.0 / (1.0 + exp_value)
    exp_value = math.exp(value)
    return exp_value / (1.0 + exp_value)


def generate_rows(seed=SEED):
    """產生 10 個月份、每月 30 筆的純合成資料。"""
    rng = random.Random(seed)
    rows = []
    contact_weights = (35, 30, 20, 10, 5)

    for month in range(10):
        if month < 6:
            premium_mean = 0.05
        elif month < 8:
            premium_mean = 0.10
        else:
            premium_mean = 0.16

        for _ in range(30):
            premium_change = min(
                0.35, max(0.0, rng.gauss(premium_mean, 0.045))
            )
            contacts_90d = rng.choices(range(5), weights=contact_weights, k=1)[0]
            tenure_years = rng.uniform(0.5, 15.0)
            logit = (
                -2.0
                + 6.0 * premium_change
                + 0.75 * contacts_90d
                - 0.12 * tenure_years
            )
            churn_probability = sigmoid(logit)
            churn = int(rng.random() < churn_probability)
            rows.append(
                {
                    "month": month,
                    "premium_change": premium_change,
                    "contacts_90d": contacts_90d,
                    "tenure_years": tenure_years,
                    "churn": churn,
                }
            )

    return rows


def split_rows(rows):
    """依時間原順序切分訓練、驗證與測試資料，不洗牌。"""
    return rows[:180], rows[180:240], rows[240:]


def fit_standardizer(train_rows):
    """僅用傳入的訓練列估計母體平均數與標準差。"""
    if not train_rows:
        raise ValueError("訓練資料不可為空")

    means = []
    stds = []
    for feature in FEATURES:
        values = [float(row[feature]) for row in train_rows]
        mean = sum(values) / len(values)
        variance = sum((value - mean) ** 2 for value in values) / len(values)
        std = math.sqrt(variance)
        means.append(mean)
        stds.append(std if std != 0.0 else 1.0)
    return means, stds


def transform_rows(rows, means, stds):
    """用訓練集統計量標準化特徵，保留其餘欄位供追蹤與評估。"""
    if len(means) != len(FEATURES) or len(stds) != len(FEATURES):
        raise ValueError("標準化統計量必須對應三個特徵")

    transformed = []
    for row in rows:
        scaled = dict(row)
        for index, feature in enumerate(FEATURES):
            scaled[feature] = (float(row[feature]) - means[index]) / stds[index]
        transformed.append(scaled)
    return transformed


def fit_logistic_regression(train_rows, steps=TRAIN_STEPS,
                            learning_rate=LEARNING_RATE, l2=L2):
    """以全批次梯度下降訓練邏輯斯迴歸；L2 不懲罰截距。"""
    if not train_rows:
        raise ValueError("訓練資料不可為空")
    if steps <= 0 or learning_rate <= 0 or l2 < 0:
        raise ValueError("steps 與 learning_rate 必須為正，l2 不可為負")

    weights = [0.0] * len(FEATURES)
    bias = 0.0
    row_count = len(train_rows)

    for _ in range(steps):
        bias_gradient = 0.0
        weight_gradients = [0.0] * len(FEATURES)
        for row in train_rows:
            score = bias + sum(
                weights[index] * float(row[feature])
                for index, feature in enumerate(FEATURES)
            )
            error = sigmoid(score) - int(row["churn"])
            bias_gradient += error
            for index, feature in enumerate(FEATURES):
                weight_gradients[index] += error * float(row[feature])

        bias -= learning_rate * bias_gradient / row_count
        for index in range(len(FEATURES)):
            gradient = weight_gradients[index] / row_count + l2 * weights[index]
            weights[index] -= learning_rate * gradient

    return bias, weights


def predict_probabilities(rows, bias, weights):
    """對標準化後的資料計算流失機率。"""
    if len(weights) != len(FEATURES):
        raise ValueError("模型權重必須對應三個特徵")
    return [
        sigmoid(
            bias
            + sum(
                weights[index] * float(row[feature])
                for index, feature in enumerate(FEATURES)
            )
        )
        for row in rows
    ]


def evaluate(y_true, probabilities, threshold):
    """計算混淆矩陣（TP/FP/FN/TN）、分類指標與 Brier score。"""
    if len(y_true) != len(probabilities) or not y_true:
        raise ValueError("標籤與機率必須等長且不可為空")
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("分類閾值必須介於 0 與 1")

    tp = fp = fn = tn = 0
    for label, probability in zip(y_true, probabilities):
        if not 0.0 <= probability <= 1.0:
            raise ValueError("預測機率必須介於 0 與 1")
        predicted = probability >= threshold
        actual = int(label) == 1
        if predicted and actual:
            tp += 1
        elif predicted:
            fp += 1
        elif actual:
            fn += 1
        else:
            tn += 1

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2.0 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )
    brier = sum(
        (float(probability) - int(label)) ** 2
        for label, probability in zip(y_true, probabilities)
    ) / len(y_true)
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "brier": brier,
    }


def choose_threshold(y_true, probabilities, thresholds=THRESHOLDS):
    """依驗證集 F1 選閾值；F1 相同時採較高閾值。"""
    if not thresholds:
        raise ValueError("閾值候選不可為空")
    candidates = [
        (float(threshold), evaluate(y_true, probabilities, float(threshold)))
        for threshold in thresholds
    ]
    return max(candidates, key=lambda candidate: (candidate[1]["f1"], candidate[0]))


def run_analysis():
    """執行完整合成資料流程並回傳可列印結果。"""
    rows = generate_rows()
    train_rows, valid_rows, test_rows = split_rows(rows)
    means, stds = fit_standardizer(train_rows)
    scaled_train = transform_rows(train_rows, means, stds)
    scaled_valid = transform_rows(valid_rows, means, stds)
    scaled_test = transform_rows(test_rows, means, stds)

    bias, weights = fit_logistic_regression(scaled_train)
    valid_y = [row["churn"] for row in valid_rows]
    valid_probabilities = predict_probabilities(scaled_valid, bias, weights)
    threshold, valid_metrics = choose_threshold(valid_y, valid_probabilities)

    test_y = [row["churn"] for row in test_rows]
    test_probabilities = predict_probabilities(scaled_test, bias, weights)
    model_metrics = evaluate(test_y, test_probabilities, threshold)

    baseline_probability = sum(row["churn"] for row in train_rows) / len(train_rows)
    baseline_metrics = evaluate(
        test_y,
        [baseline_probability] * len(test_y),
        0.5,
    )
    return {
        "rows": rows,
        "train_rows": train_rows,
        "valid_rows": valid_rows,
        "test_rows": test_rows,
        "means": means,
        "stds": stds,
        "weights": weights,
        "bias": bias,
        "threshold": threshold,
        "valid_metrics": valid_metrics,
        "baseline_probability": baseline_probability,
        "baseline_metrics": baseline_metrics,
        "model_metrics": model_metrics,
        "test_average_premium_change": sum(
            row["premium_change"] for row in test_rows
        ) / len(test_rows),
    }


def print_metrics(name, result):
    print(
        f"{name}: TP={result['tp']} FP={result['fp']} "
        f"FN={result['fn']} TN={result['tn']} | "
        f"precision={result['precision']:.3f} "
        f"recall={result['recall']:.3f} F1={result['f1']:.3f} "
        f"Brier={result['brier']:.3f}"
    )


def print_report(result):
    train_rows = result["train_rows"]
    valid_rows = result["valid_rows"]
    test_rows = result["test_rows"]
    print(
        "合成虛構資料：seed=20261007，資料筆數="
        f"{len(result['rows'])}，train/valid/test="
        f"{len(train_rows)}/{len(valid_rows)}/{len(test_rows)}"
    )
    print(
        "正例數：train="
        f"{sum(row['churn'] for row in train_rows)}，valid="
        f"{sum(row['churn'] for row in valid_rows)}，test="
        f"{sum(row['churn'] for row in test_rows)}"
    )
    print(
        "test 平均 premium_change="
        f"{result['test_average_premium_change']:.3f}；"
        f"validation 最佳 F1 閾值={result['threshold']:.2f} "
        f"(validation F1={result['valid_metrics']['f1']:.3f})"
    )
    print(
        "Baseline（訓練集正例率="
        f"{result['baseline_probability']:.3f}，分類閾值=0.50）"
    )
    print_metrics("  測試集", result["baseline_metrics"])
    print(f"Logistic regression（分類閾值={result['threshold']:.2f}）")
    print_metrics("  測試集", result["model_metrics"])
    print("注意：以上只展示合成資料的示範結果，不代表真實續保效益。")


def main():
    print_report(run_analysis())


In [2]:
results = run_analysis()
print_report(results)

合成虛構資料：seed=20261007，資料筆數=300，train/valid/test=180/60/60
正例數：train=32，valid=18，test=12
test 平均 premium_change=0.165；validation 最佳 F1 閾值=0.25 (validation F1=0.522)
Baseline（訓練集正例率=0.178，分類閾值=0.50）
  測試集: TP=0 FP=0 FN=12 TN=48 | precision=0.000 recall=0.000 F1=0.000 Brier=0.160
Logistic regression（分類閾值=0.25）
  測試集: TP=12 FP=32 FN=0 TN=16 | precision=0.273 recall=1.000 F1=0.429 Brier=0.157
注意：以上只展示合成資料的示範結果，不代表真實續保效益。
